# 00c. pandas III: time series

Almost every frame in this course has dates for its index. This notebook covers the
time-series tools and, above all, **which of them look into the future**. Getting that
wrong doesn't raise an error; it just gives you a backtest that looks better than it
should.

**How to work through this.** Before running a cell, say out loud what you expect it to
print, including the *shape* and the *labels*. Most pandas confusion is a wrong guess about
which labels come back, and you only notice if you guessed first. Exercises have an empty
cell beneath them and the expected answer above; match it, then move on.

## Setup

Synthetic prices from `portfolio_lab.data.synthetic`: the same eight tickers as week 1,
business days from 2015-01-01, no network and no `data/` needed, identical on every run
(`seed=7`). The generator returns the store's **long** format; `pivot` turns it into the
**wide** frame the course uses everywhere. Primer 00d explains `pivot`; for now take it on
trust.

The generator's drift is deliberately strong (so the momentum tests have something to
find), so price **levels** wander far from anything real: some tickers end at a fraction of
a cent on the dollar. Use these frames for mechanics, and don't read finance into the numbers.

In [1]:
import numpy as np
import pandas as pd

from portfolio_lab.data.returns import simple_returns
from portfolio_lab.data.synthetic import make_synthetic_prices

pd.set_option("display.precision", 4)
pd.set_option("display.width", 100)

TICKERS = ["SPY", "EFA", "IEF", "GLD", "AAPL", "MSFT", "JNJ", "XOM"]
long, truth = make_synthetic_prices(TICKERS, n_days=2520, seed=7)  # long format, like the store
long["date"] = pd.to_datetime(long["date"])
px = long.pivot(index="date", columns="ticker", values="close")    # wide: dates x tickers
px.columns.name = None
rets = simple_returns(px)                                           # your week-1 function
px.shape, rets.shape

((2520, 8), (2520, 8))

## 1. The DatetimeIndex

A `DatetimeIndex` gives you partial-string selection (00a), date components, and
calendar-aware resampling. Get one by converting with `pd.to_datetime`, which the setup
cell did.

In [2]:
idx = px.index
print(type(idx).__name__, idx.min().date(), idx.max().date(), len(idx))
print(idx[:3].year.tolist(), idx[:3].month.tolist(), idx[:3].day_name().tolist())

DatetimeIndex 2015-01-01 2024-08-28 2520
[2015, 2015, 2015] [1, 1, 1] ['Thursday', 'Friday', 'Monday']


The synthetic calendar is every business day, holidays included. Real exchange calendars
skip holidays, so on real data **never assume 252 rows is exactly one year**, and never
assume two tickers share a calendar until you've checked.

In [3]:
wk = pd.bdate_range(idx.min(), idx.max())
print(len(wk), "weekdays in range;", len(idx), "rows in px")

2520 weekdays in range; 2520 rows in px


## 2. Shifting: the direction that matters

`shift(1)` moves values **down** a row: each date sees **yesterday's** value. That is the
past, and it's safe. `shift(-1)` moves values **up**: each date sees **tomorrow's** value.
That is the canonical lookahead bug, and the reason `tests/test_no_lookahead.py` exists.

In [4]:
pd.DataFrame({"SPY": px["SPY"], "shift(1) past": px["SPY"].shift(1),
              "shift(-1) FUTURE": px["SPY"].shift(-1)}).head(4)

,SPY,shift(1) past,shift(-1) FUTURE
date,,,
2015-01-01,100.7528,NaN,97.9358
2015-01-02,97.9358,100.7528,99.0545
2015-01-05,99.0545,97.9358,96.9184
2015-01-06,96.9184,99.0545,94.7434


The family:

| Expression | Meaning |
|---|---|
| `px.shift(1)` | yesterday's value on today's row |
| `px.diff()` | `px - px.shift(1)` |
| `px.pct_change()` | `px / px.shift(1) - 1` (pandas 3: no silent fill across gaps) |
| `np.log(px).diff()` | log returns |
| `px / px.shift(21) - 1` | 21-day (about one month) return, ending today |
| `px.shift(21) / px.shift(252) - 1` | the 12-1 momentum signal: 12-month return, skipping the latest month |

**Signal timing.** A signal computed from data *through* day t can only be traded at the
*next* open. The engine enforces that with `pending` (see `CLAUDE.md`). If you compute a
signal and returns on the same row and then multiply them, you've traded on information
you didn't have yet. Lag the signal: `signal.shift(1) * rets`.

## 3. Cumulative: equity curves and drawdowns

In [5]:
growth = (1 + rets).cumprod()                 # $1 invested at the start
growth.iloc[0] = 1.0                          # row 0 was NaN: nothing earned yet
log_growth = np.exp(np.log(px).diff().cumsum())
print(np.allclose(growth.iloc[1:], log_growth.iloc[1:]))   # the two identities agree
print(np.allclose(growth, px / px.iloc[0]))                # ...and equal price relatives
growth.iloc[-1].round(3)

True
True


AAPL     1.582
EFA      6.139
GLD      0.007
IEF      0.355
JNJ      0.109
MSFT    16.279
SPY      0.063
XOM      0.311
Name: 2024-08-28 00:00:00, dtype: float64

**Drawdown** is the loss from the running peak, and it's the first risk number anyone
asks about. `cummax` gives the running peak, so drawdown takes one line:

In [6]:
peak = growth.cummax()
dd = growth / peak - 1
dd[["SPY", "IEF"]].describe().loc[["min", "mean"]]

,SPY,IEF
min,-0.9685,-0.9554
mean,-0.6951,-0.6083


**Exercise 1.** For each ticker: the maximum drawdown (most negative `dd`) and the date
it hit bottom. Answer with a DataFrame with columns `max_dd` and `trough`, one row per
ticker, sorted by `max_dd` (worst first). Round `max_dd` to 4 places.

Expected:

```
      max_dd     trough
GLD  -0.9958 2021-10-11
SPY  -0.9685 2024-02-13
AAPL -0.9627 2024-07-30
IEF  -0.9554 2021-11-23
JNJ  -0.9477 2024-03-28
XOM  -0.9461 2024-08-28
MSFT -0.7846 2024-08-05
EFA  -0.4820 2018-11-20
```

## 4. Rolling windows

`rolling(n)` looks at the **trailing** n rows ending at each date, including that date.
Trailing means causal: the value on day t uses only days up to t.

In [7]:
vol63 = rets.rolling(63).std() * np.sqrt(252)     # ~3-month trailing annualised vol
print(vol63["SPY"].isna().sum(), "NaN: row 0 has no return, then 62 rows to fill the window")
vol63[["SPY", "IEF"]].dropna().head(3)

63 NaN: row 0 has no return, then 62 rows to fill the window


,SPY,IEF
date,,
2015-03-31,0.2266,0.2152
2015-04-01,0.2198,0.2154
2015-04-02,0.2191,0.2155


The options that matter:

| Option | Effect | Lookahead? |
|---|---|---|
| `rolling(63)` | window of 63 rows, NaN until full | no |
| `rolling(63, min_periods=20)` | start producing values once 20 are present | no |
| `rolling(63, center=True)` | window **centred** on t: 31 days either side | **yes** |
| `rolling("90D")` | time-based window: the last 90 calendar days | no |
| `ewm(halflife=21)` | exponentially weighted: recent days count more | no |
| `expanding()` | everything from the start up to t | no |

`ewm` is the standard risk-model choice (RiskMetrics used it). You'll meet it in week 4.

In [8]:
ewm_vol = rets["SPY"].ewm(halflife=21).std() * np.sqrt(252)
pd.DataFrame({"rolling63": vol63["SPY"], "ewm_hl21": ewm_vol}).iloc[[100, 1000, 2000]]

,rolling63,ewm_hl21
date,,
2015-05-21,0.2266,0.2301
2018-11-01,0.2279,0.2262
2022-09-01,0.2268,0.2192


`rolling(...).cov()` and `.corr()` give moving covariances. Pair two columns like this:

In [9]:
rets["SPY"].rolling(252).corr(rets["IEF"]).dropna().describe()[["mean", "min", "max"]]

mean    0.4519
min     0.3044
max     0.5304
dtype: float64

**Exercise 2.** 63-day trailing annualised vol for SPY, using the `rolling` form. Answer
with its value on the last date, rounded to 6 places. Then **prove it's causal**: compute
it again on `rets.loc[:"2020-12-31"]` only, and assert the two agree on every date both
have. (This is the first half of `test_no_lookahead.py`, done by hand.)

Expected:

```
0.202043
```

**Exercise 3.** Now break it. Repeat the truncation test with `center=True`. On how many
dates do the full-history and truncated versions **disagree**? (Compare with
`np.isclose`, and count a NaN on one side against a number on the other as a
disagreement.) The count is the size of the lookahead: think about why it's that number.

Expected:

```
31
```

## 5. Resampling: changing frequency

`resample(rule)` groups rows into calendar buckets, like a `groupby` on time. The rules
you'll use: `"ME"` month-end, `"QE"` quarter-end, `"YE"` year-end, `"W-FRI"` weeks ending
Friday. (pandas 2.2 renamed `"M"`/`"Y"` to `"ME"`/`"YE"`; older tutorials use the old
names.)

There are two correct ways to get monthly returns. Knowing both is a free check:

In [10]:
m1 = (1 + rets).resample("ME").prod() - 1          # compound the days in each month
m2 = px.resample("ME").last().pct_change()          # month-end price over month-end price
print(np.allclose(m1.iloc[1:], m2.iloc[1:]))        # same, except the first month
m1[["SPY", "IEF"]].head(3)

True


,SPY,IEF
date,,
2015-01-31,0.0182,0.0755
2015-02-28,0.0352,-0.0541
2015-03-31,0.0628,-0.1011


`resample` labels each bucket by the **calendar** period end (e.g. 2015-01-31, a Saturday),
not the last trading day in it. That's fine for monthly statistics; it matters if you try
to join monthly results back onto trading days (use `merge_asof`, 00d).

Note what `prod` is doing: compounding. **Never** sum simple returns across time;
`(1 + r).prod() - 1` or sum the **log** returns. That's the asymmetry from your lesson 1.

**Exercise 4.** Calendar-year returns: a table with **years as rows** and tickers as
columns, rounded to 4 places. Label the rows with plain integers (2015, 2016, ...) rather
than timestamps. Show the first three years. (`resample("YE")` then `.index.year`.)

Expected:

```
        AAPL     EFA     GLD     IEF     JNJ    MSFT     SPY     XOM
date                                                                
2015  0.4039 -0.2939 -0.2176  0.9023 -0.0441 -0.4604  0.1672  0.5651
2016  0.9174  1.7469 -0.4933  0.1565 -0.1144  0.6408 -0.0348  0.1966
2017  2.0757  0.6697 -0.6582 -0.0385  0.6615  1.4719 -0.7007  1.6566
```

**Exercise 5.** The 12-1 momentum signal: `px.shift(21) / px.shift(252) - 1`. Then rank it
**across tickers on each date** (`rank(axis=1)`, 1 = weakest). Answer with the ranks on
the last date as integers, sorted from strongest down. This is the shape of a week 6
signal: one score per ticker per date, using only the past.

Expected:

```
SPY     8
JNJ     7
EFA     6
AAPL    5
IEF     4
GLD     3
XOM     2
MSFT    1
Name: 2024-08-28 00:00:00, dtype: int64
```

## 6. The lookahead checklist

Each of these runs without an error and produces plausible numbers:

| Construct | Why it leaks the future |
|---|---|
| `shift(-k)` | pulls a future row onto today |
| `bfill()`, `interpolate()` | fills today from a later observation |
| `rolling(..., center=True)` | the window reaches forward |
| `x - x.mean()`, `x / x.std()` over the **whole sample** | today's value uses the whole future sample's mean |
| fitting a model (regression, covariance) on all dates, then backtesting | same: full-sample fit |
| signal and return on the same row, multiplied | trades on the close you just observed |
| today's index members on past dates | survivorship: week 5 |

The defence is the test you did in Exercise 2: **truncate the history, and nothing
already computed may change.** The course's `test_no_lookahead.py` does exactly that to
every signal.

## Reading for this notebook

- McKinney, *Python for Data Analysis* 3e, ch. 11 "Time Series" (date ranges, shifting,
  resampling, moving-window functions): https://wesmckinney.com/book/
- VanderPlas, *Python Data Science Handbook*, "Working with Time Series":
  https://jakevdp.github.io/PythonDataScienceHandbook/
- pandas user guide, "Time series / date functionality" and "Windowing operations":
  https://pandas.pydata.org/docs/user_guide/timeseries.html and
  https://pandas.pydata.org/docs/user_guide/window.html
- Lopez de Prado, *Advances in Financial Machine Learning*, ch. 7, for why leakage is the
  central risk in financial ML (already on the course reading list).